[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/04-rnns-lstms.ipynb)

# RNNs and LSTMs
**Module 7 — Lesson 4 | Estimated time: 35 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Explain why sequence models differ from feedforward networks
- Implement a vanilla RNN cell from scratch
- Demonstrate vanishing gradients with gradient norm plots
- Describe LSTM gate equations (forget/input/output/cell)
- Use `nn.RNN`, `nn.LSTM`, and `nn.GRU` APIs
- Train a character-level language model and generate text

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. Sequence Modelling Motivation

Feedforward networks assume inputs are independent. Sequences — text, audio, time series — have **temporal dependencies** where earlier tokens affect later meanings.

Examples:
- `"The bank by the river"` vs `"The bank approved the loan"` — "bank" meaning depends on context
- Stock price tomorrow depends on the last N days
- Next character in `"happ"` is likely `"y"` or `"i"`

In [ ]:
# Visualise a simple time series dependency
t = np.linspace(0, 4*np.pi, 200)
y = np.sin(t) + 0.1 * np.random.randn(200)

plt.figure(figsize=(10, 3))
plt.plot(t, y, label='sin(t) + noise')
plt.axvspan(t[50], t[70], alpha=0.3, color='red', label='Context window')
plt.axvline(t[70], color='red', ls='--', label='Prediction point')
plt.title('Sequence Modelling: predict next value from history')
plt.legend(); plt.tight_layout(); plt.show()

## 2. Vanilla RNN from Scratch

An RNN maintains a hidden state `h_t` updated at each time step:

$$h_t = \tanh(W_{hh}\, h_{t-1} + W_{xh}\, x_t + b_h)$$

The same weights are shared across all time steps (parameter sharing).

In [ ]:
class VanillaRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_xh = nn.Linear(input_size,  hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)

    def forward(self, x_t, h_prev):
        return torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))

class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell   = VanillaRNNCell(input_size, hidden_size)
        self.fc_out = nn.Linear(hidden_size, output_size)

    def forward(self, x):  # x: (batch, seq_len, input_size)
        batch, seq_len, _ = x.shape
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        for t in range(seq_len):
            h = self.cell(x[:, t, :], h)
        return self.fc_out(h)  # many-to-one

# Quick test
rnn_scratch = VanillaRNN(input_size=5, hidden_size=16, output_size=3)
x_test = torch.randn(4, 10, 5)  # batch=4, seq=10, features=5
out = rnn_scratch(x_test)
print('RNN scratch output:', out.shape)  # (4, 3)

## 3. Vanishing Gradient Problem

In a deep unrolled RNN, gradients are multiplied by `W_hh` at each step. If the largest eigenvalue of `W_hh` is < 1, gradients **vanish exponentially** with sequence length.

In [ ]:
def track_gradient_norms(model_type='rnn', seq_len=50, hidden=32):
    """Track gradient norm at each time step (backprop through time)."""
    if model_type == 'rnn':
        cell = nn.RNNCell(1, hidden)
    else:
        cell = nn.LSTMCell(1, hidden)

    x_seq = torch.randn(seq_len, 1, 1)  # (seq, batch=1, features=1)
    h = torch.zeros(1, hidden)
    c = torch.zeros(1, hidden)

    grad_norms = []
    outputs = []
    for t in range(seq_len):
        if model_type == 'rnn':
            h = cell(x_seq[t], h)
            outputs.append(h)
        else:
            h, c = cell(x_seq[t], (h, c))
            outputs.append(h)

    # Compute gradient of final output w.r.t. each step's output
    final = outputs[-1].sum()
    for out in outputs:
        if out.grad_fn is not None:
            g = torch.autograd.grad(final, out, retain_graph=True, allow_unused=True)
            norm = g[0].norm().item() if g[0] is not None else 0.0
        else:
            norm = 0.0
        grad_norms.append(norm)
    return grad_norms

rnn_norms  = track_gradient_norms('rnn',  seq_len=30)
lstm_norms = track_gradient_norms('lstm', seq_len=30)

plt.figure(figsize=(9, 4))
plt.plot(rnn_norms,  label='Vanilla RNN', color='red',      lw=2)
plt.plot(lstm_norms, label='LSTM',        color='steelblue', lw=2)
plt.xlabel('Time step'); plt.ylabel('Gradient norm')
plt.title('Gradient Norms Through Time: RNN vs LSTM')
plt.legend(); plt.tight_layout(); plt.show()

## 4. LSTM Cell — Gate Equations

LSTM adds a **cell state** `c_t` as a highway for gradients, controlled by three gates:

| Gate | Equation | Purpose |
|------|----------|--------|
| Forget | `f = σ(W_f·[h,x] + b_f)` | How much of old cell state to keep |
| Input  | `i = σ(W_i·[h,x] + b_i)` | What new info to write |
| Output | `o = σ(W_o·[h,x] + b_o)` | What to expose as hidden state |
| Cell   | `g = tanh(W_g·[h,x] + b_g)` | Candidate cell values |

Update rules:
```
c_t = f ⊙ c_{t-1} + i ⊙ g
h_t = o ⊙ tanh(c_t)
```

In [ ]:
# PyTorch nn.LSTM API
rnn  = nn.RNN( input_size=8, hidden_size=32, num_layers=2, batch_first=True, dropout=0.2)
lstm = nn.LSTM(input_size=8, hidden_size=32, num_layers=2, batch_first=True, dropout=0.2)
gru  = nn.GRU( input_size=8, hidden_size=32, num_layers=2, batch_first=True, dropout=0.2)

x = torch.randn(16, 20, 8)  # batch=16, seq=20, features=8

out_rnn,  h_rnn          = rnn(x)
out_lstm, (h_lstm, c_lstm) = lstm(x)
out_gru,  h_gru          = gru(x)

print('RNN  output:', out_rnn.shape,  ' hidden:', h_rnn.shape)
print('LSTM output:', out_lstm.shape, ' hidden:', h_lstm.shape, ' cell:', c_lstm.shape)
print('GRU  output:', out_gru.shape,  ' hidden:', h_gru.shape)

# Parameter count comparison
for name, m in [('RNN', rnn), ('LSTM', lstm), ('GRU', gru)]:
    p = sum(p.numel() for p in m.parameters())
    print(f'{name}: {p:,} parameters')

## 5. Character-Level Language Model

Train a character-level LSTM to predict the next character in a sequence, then use it to generate text.

In [ ]:
# Small training corpus
text = (
    "to be or not to be that is the question whether tis nobler in the mind to suffer "
    "the slings and arrows of outrageous fortune or to take arms against a sea of troubles "
    "and by opposing end them to die to sleep no more and by a sleep to say we end "
    "the heartache and the thousand natural shocks that flesh is heir to tis a consummation "
    "devoutly to be wished to die to sleep to sleep perchance to dream ay there's the rub "
) * 4  # repeat to have more data

chars   = sorted(set(text))
vocab   = {ch: i for i, ch in enumerate(chars)}
id2char = {i: ch for ch, i in vocab.items()}
V       = len(chars)
print(f'Vocab size: {V}, Text length: {len(text)}')

# Build sequences
SEQ_LEN = 40
X_list, y_list = [], []
for i in range(0, len(text) - SEQ_LEN - 1, 3):
    seq  = [vocab[c] for c in text[i:i+SEQ_LEN]]
    target = vocab[text[i+SEQ_LEN]]
    X_list.append(seq)
    y_list.append(target)

X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)
print(f'Sequences: {X_t.shape}, Targets: {y_t.shape}')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden=128, num_layers=2):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim)
        self.lstm   = nn.LSTM(embed_dim, hidden, num_layers, batch_first=True, dropout=0.3)
        self.fc     = nn.Linear(hidden, vocab_size)

    def forward(self, x, state=None):
        x = self.embed(x)
        out, state = self.lstm(x, state)
        return self.fc(out[:, -1, :]), state  # many-to-one

model     = CharLSTM(V).to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(X_t, y_t)
dl      = DataLoader(dataset, batch_size=128, shuffle=True)

losses = []
for epoch in range(12):
    model.train()
    epoch_loss = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad()
        out, _ = model(xb)
        loss   = criterion(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimiser.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(dl)
    losses.append(avg)
    if epoch % 3 == 0:
        print(f'Epoch {epoch+1:2d}: loss={avg:.3f}')

plt.plot(losses); plt.title('Character LM Training Loss')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy'); plt.show()

## 6. Text Generation Inference

In [ ]:
def generate_text(model, seed, length=200, temperature=0.8):
    model.eval()
    result = seed
    state  = None
    # Warm up on seed
    inp = torch.tensor([[vocab[c] for c in seed]], dtype=torch.long).to(device)
    with torch.no_grad():
        _, state = model(inp)
    # Generate char by char
    last_char = seed[-1]
    for _ in range(length):
        inp = torch.tensor([[vocab[last_char]]], dtype=torch.long).to(device)
        with torch.no_grad():
            logits, state = model(inp, state)
        probs = torch.softmax(logits / temperature, dim=-1).squeeze().cpu().numpy()
        idx   = np.random.choice(V, p=probs)
        last_char = id2char[idx]
        result += last_char
    return result

generated = generate_text(model, seed='to be or not', length=300, temperature=0.7)
print('Generated text:')
print('-' * 60)
print(generated)

## Practice Exercises

**Exercise 1 — GRU Comparison**
Swap the LSTM in `CharLSTM` for a GRU (`nn.GRU`). Train both for the same number of epochs, compare parameter counts and final perplexity (`exp(loss)`), and generate sample text from each.

**Exercise 2 — Many-to-Many: Sequence Labelling**
Modify the LSTM model to return outputs at every time step (remove `[:, -1, :]`). Use it for a simple POS-tagging task: label each character as vowel (1) or consonant (0) in a string. Train and report per-token accuracy.

**Exercise 3 — Bidirectional LSTM**
Add `bidirectional=True` to `nn.LSTM`. Adjust the classifier `nn.Linear` input size accordingly (hidden × 2). Train on a sentiment classification task using random movie review snippets and report accuracy.